# MoMoK — MKG-W Reproduction
## ICLR 2025

This notebook reproduces the RQ1 main full-model MKG-W experiment from the official MoMoK repository, pinned to commit `99c2df114d48c79708ea0608644b685183d81bd9`. The configuration source is the official MoMoK README. This is the full model only: no ablation and no PMF.

Official precomputed embeddings are used; raw feature extraction is not performed. The notebook validates the published MKG-W contract, runs a bounded real-data smoke test, then executes exactly one 2000-epoch full run.

Paper target: **MRR = 0.3589**, **Hits@1 = 0.3038**, **Hits@3 = 0.3754**, **Hits@10 = 0.4613**.

In [ ]:
REPO_URL = "https://github.com/zjukg/MoMoK.git"
REPO_COMMIT = "99c2df114d48c79708ea0608644b685183d81bd9"
WORK_ROOT = "/kaggle/working"
REPO_DIR = "/kaggle/working/MoMoK"
OUTPUT_DIR = "/kaggle/working/repro_output"
DATASET = "MKG-W"
CONFIG_SOURCE = "OFFICIAL_README"
GOOGLE_DRIVE_FILE_ID = "1dKJdJunb11kDtFr5NLfPlFknS7cRdm9W"
ACTIVE_KERNEL_REF = "nientrandai1/momok-mkg-w-reproduction"
print("ACTIVE_KERNEL_REF=" + ACTIVE_KERNEL_REF)
print("NO_ACTIVE_DB15K_KERNEL_REF=YES")
FEATURE_ARCHIVE_MEMBERS = {'img_features.pth': 'embeddings/MKG-W/img_features.pth', 'text_features.pth': 'embeddings/MKG-W/text_features.pth'}
PAPER_METRICS = {'MRR': 0.3589, 'Hits@1': 0.3038, 'Hits@3': 0.3754, 'Hits@10': 0.4613}
EXPECTED_DATASET = {'entities': 15000, 'relations': 169, 'train': 34196, 'valid': 4276, 'test': 4274, 'image_available': 14463, 'image_dim': 383, 'text_available': 14123, 'text_dim': 384}
SEED = 10010
BATCH_SIZE = 1024
N_EXPERTS = 3
EPOCHS = 2000
EVAL_FREQ = 100
LR = 0.001
MU = 0.0001
DIM = 200
R_DIM = 256
RESUME_MODE = "FRESH_RESTART_WITH_CHECKPOINTING"
print("RESUME_FROM_DB15K=NO")
print("RESUME_CHECKPOINT=NONE")
print("COMPLETED_EPOCH_AT_START=0")
print("NEXT_HUMAN_EPOCH=1")
print("START_EPOCH=0")
print("TARGET_EPOCH=2000")
FEATURE_EXTRACTION_REQUIRED = False
OFFICIAL_PRECOMPUTED_FEATURES = True
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
SESSION_SOFT_LIMIT_SECONDS = 37800
RESUME_DATASET_REF = "nientrandai1/momok-mkgw-resume-state"
RESUME_MODE = "FRESH_RESTART_WITH_CHECKPOINTING"
RESUME_FROM_PREVIOUS_TARGET = False
RESUME_CHECKPOINT_PATH = None


In [ ]:
import json, os, platform, shutil, subprocess, sys
from pathlib import Path

ram_gb = None
try:
    import psutil
    ram_gb = round(psutil.virtual_memory().total / (1024**3), 2)
except Exception:
    pass

def run_capture(args, **kwargs):
    return subprocess.run(args, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, **kwargs)

print("platform=", platform.platform())
print("python=", sys.version)
print("cpu_count=", os.cpu_count())
print("ram_gb=", ram_gb)
print("disk_usage=", shutil.disk_usage(WORK_ROOT))
print("kaggle_dirs=", sorted(str(p) for p in Path("/kaggle").glob("*")))
sm = run_capture(["bash", "-lc", "nvidia-smi || true"])
print(sm.stdout)
Path(OUTPUT_DIR, "runtime_preinstall.json").write_text(json.dumps({
    "platform": platform.platform(), "python": sys.version, "cpu_count": os.cpu_count(),
    "ram_gb": ram_gb, "disk_usage": dict(zip(["total", "used", "free"], shutil.disk_usage(WORK_ROOT))),
    "nvidia_smi": sm.stdout, "kaggle_dirs": sorted(str(p) for p in Path("/kaggle").glob("*"))
}, indent=2))

In [ ]:
import importlib.util, subprocess, sys

packages = ["numpy", "scikit-learn", "tqdm", "h5py", "gdown", "wandb"]
install_args = [sys.executable, "-m", "pip", "install", "-q"] + packages
subprocess.check_call(install_args)
try:
    import importlib.metadata as metadata
    print("torch_before_import=", metadata.version("torch"))
except Exception as exc:
    print("torch_before_import=unknown", repr(exc))
print("Dependencies installed in the running kernel environment. Kaggle's GPU-compatible torch is retained.")

In [ ]:
import h5py, numpy, sklearn, torch

assert torch.cuda.is_available(), "Kaggle GPU is required for this reproduction"
gpu_name = torch.cuda.get_device_name(0)
print("python=", sys.version.split()[0])
print("torch=", torch.__version__)
print("torch_cuda=", torch.version.cuda)
print("gpu=", gpu_name)
print("numpy=", numpy.__version__)
print("sklearn=", sklearn.__version__)
print("h5py=", h5py.__version__)
environment = {
    "paper_python": "3.9", "paper_torch": "2.0.0",
    "effective_python": sys.version.split()[0], "effective_torch": torch.__version__,
    "effective_cuda": torch.version.cuda, "gpu": gpu_name,
    "runtime_deviation": "Kaggle-compatible runtime retained; torch was not forcibly downgraded."
}
Path(OUTPUT_DIR, "environment.json").write_text(json.dumps(environment, indent=2))

In [ ]:
import shutil, subprocess

if Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)
subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
subprocess.check_call(["git", "-C", REPO_DIR, "checkout", "--detach", REPO_COMMIT])
actual_commit = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
assert actual_commit == REPO_COMMIT, (actual_commit, REPO_COMMIT)
print("PINNED_COMMIT=" + actual_commit)
Path(OUTPUT_DIR, "source.json").write_text(json.dumps({"repo_url": REPO_URL, "commit": actual_commit, "status": "detached"}, indent=2))

In [ ]:
import difflib, json, os, re

train_path = Path(REPO_DIR, "train.py")
loader_path = Path(REPO_DIR, "utils", "data_loader.py")
original_train = train_path.read_text()
original_loader = loader_path.read_text()
fixes = []
train_text = original_train
loader_text = original_loader

if "return metrics, [att_s, att_i, att_t, att_mm]" in loader_text and "val_metrics = corpus.get_validation_pred(model, 'test')" in train_text:
    train_text = train_text.replace("val_metrics = corpus.get_validation_pred(model, 'test')", "val_metrics, _ = corpus.get_validation_pred(model, 'test')")
    fixes.append({"file": "train.py", "before": "val_metrics = corpus.get_validation_pred(model, 'test')", "after": "val_metrics, _ = corpus.get_validation_pred(model, 'test')", "reason": "upstream validation returns (metrics, attention) while train.py indexes it as a dict", "semantic_change": False})

if "y.byte()" in loader_text:
    loader_text = loader_text.replace("y.byte()", "y.bool()")
    fixes.append({"file": "utils/data_loader.py", "before": "y.byte()", "after": "y.bool()", "reason": "modern PyTorch boolean-mask compatibility", "semantic_change": False})

if "torch.save(model.state_dict(), f'./checkpoint/{args.dataset}/{args.model}.pth')" in train_text and "os.makedirs('./checkpoint/'" not in train_text:
    train_text = train_text.replace("import argparse\n", "import argparse\nimport os\n")
    train_text = train_text.replace("    if args.save:\n        torch.save(model.state_dict(), f'./checkpoint/{args.dataset}/{args.model}.pth')", "    if args.save:\n        os.makedirs(f'./checkpoint/{args.dataset}', exist_ok=True)\n        torch.save(model.state_dict(), f'./checkpoint/{args.dataset}/{args.model}.pth')")
    fixes.append({"file": "train.py", "before": "torch.save without parent directory creation", "after": "os.makedirs(..., exist_ok=True) before torch.save", "reason": "Kaggle output path may not exist", "semantic_change": False})

if train_text != original_train:
    train_path.write_text(train_text)
if loader_text != original_loader:
    loader_path.write_text(loader_text)
if 'resume_checkpoint' not in train_text:
    train_text = train_text.replace('import argparse\n', 'import argparse\nimport copy\nimport random\n')
    train_text = train_text.replace("'save': 1,", "'save': 1,\n        'resume_checkpoint': '',\n        'checkpoint_every': 25,\n        'checkpoint_dir': os.environ.get('MOMOK_CHECKPOINT_DIR', '/kaggle/working/repro_output/checkpoints'),")
    helper = '\n'.join([
        'def _move_optimizer_state(optimizer, device):',
        '    for state in optimizer.state.values():',
        '        for key, value in state.items():',
        '            if torch.is_tensor(value): state[key] = value.to(device)',
        '',
        'def _save_training_checkpoint(args, model, estimator, optimizer, optimizer_mi, lr_scheduler, best_test_metrics, corpus, completed_epoch):',
        '    os.makedirs(args.checkpoint_dir, exist_ok=True)',
        '    payload = {',
        "        'format_version': 1, 'completed_epoch': completed_epoch,",
        "        'model_state_dict': model.state_dict(), 'estimator_state_dict': estimator.state_dict(),",
        "        " + "'optimizer_state_dict': optimizer.state_dict(), 'optimizer_mi_state_dict': optimizer_mi.state_dict(),",
        "        'lr_scheduler_state_dict': lr_scheduler.state_dict(), 'best_test_metrics': best_test_metrics,",
        "        " + "'python_random_state': random.getstate(), 'numpy_rng_state': np.random.get_state(),",
        "        'torch_rng_state': torch.get_rng_state(), 'torch_cuda_rng_state_all': torch.cuda.get_rng_state_all(),",
        "        'corpus_train_indices': copy.deepcopy(corpus.train_indices),",
        "        'config': {'dataset': args.dataset, 'seed': args.seed, 'lr': args.lr, 'mu': args.mu, 'dim': args.dim, 'batch_size': args.batch_size, 'n_exp': args.n_exp, 'epochs': args.epochs, 'eval_freq': args.eval_freq, 'source_commit': os.environ.get('MOMOK_SOURCE_COMMIT', '')},"
        '    }',
        "    latest = os.path.join(args.checkpoint_dir, 'checkpoint_latest.pt')",
        "    tmp = latest + '.tmp'",
        '    with open(tmp, "wb") as handle:',
        '        torch.save(payload, handle)',
        '        handle.flush(); os.fsync(handle.fileno())',
        '    os.replace(tmp, latest)',
        "    if completed_epoch % args.eval_freq == 0:",
        "        eval_path = os.path.join(args.checkpoint_dir, 'checkpoint_eval_latest.pt')",
        "        eval_tmp = eval_path + '.tmp'",
        '        with open(eval_tmp, "wb") as handle: torch.save(payload, handle); handle.flush(); os.fsync(handle.fileno())',
        '        os.replace(eval_tmp, eval_path)',
        '    if completed_epoch == args.epochs:',
        "        final_path = os.path.join(args.checkpoint_dir, 'checkpoint_final.pt')",
        "        final_tmp = final_path + '.tmp'",
        '        with open(final_tmp, "wb") as handle: torch.save(payload, handle); handle.flush(); os.fsync(handle.fileno())',
        '        os.replace(final_tmp, final_path)',
        '    print(f"CHECKPOINT_SAVED=checkpoint_latest.pt COMPLETED_EPOCH={completed_epoch}")',
    ])
    train_text = train_text.replace('\n\ndef train_decoder(args):', '\n' + helper + '\n\ndef train_decoder(args):')
    resume_block = """    start_epoch = 0
    if args.resume_checkpoint:
        checkpoint = torch.load(args.resume_checkpoint, map_location='cpu')
        required = ['completed_epoch', 'model_state_dict', 'estimator_state_dict', 'optimizer_state_dict', 'optimizer_mi_state_dict', 'lr_scheduler_state_dict', 'best_test_metrics', 'python_random_state', 'numpy_rng_state', 'torch_rng_state', 'torch_cuda_rng_state_all', 'corpus_train_indices', 'config']
        missing = [key for key in required if key not in checkpoint]
        if missing: raise RuntimeError(f'Invalid resumable checkpoint; missing {missing}')
        expected = {'dataset': args.dataset, 'seed': args.seed, 'lr': args.lr, 'mu': args.mu, 'dim': args.dim, 'batch_size': args.batch_size, 'n_exp': args.n_exp, 'epochs': args.epochs, 'eval_freq': args.eval_freq}
        saved = checkpoint['config']
        for key, value in expected.items():
            if saved.get(key) != value: raise RuntimeError(f'Checkpoint config mismatch for {key}: {saved.get(key)} != {value}')
        model.load_state_dict(checkpoint['model_state_dict'])
        estimator.load_state_dict(checkpoint['estimator_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        optimizer_mi.load_state_dict(checkpoint['optimizer_mi_state_dict'])
        lr_scheduler.load_state_dict(checkpoint['lr_scheduler_state_dict'])
        _move_optimizer_state(optimizer, args.device)
        _move_optimizer_state(optimizer_mi, args.device)
        best_test_metrics = checkpoint['best_test_metrics']
        corpus.train_indices = copy.deepcopy(checkpoint['corpus_train_indices'])
        random.setstate(checkpoint['python_random_state'])
        np.random.set_state(checkpoint['numpy_rng_state'])
        torch.set_rng_state(checkpoint['torch_rng_state'])
        if torch.cuda.is_available(): torch.cuda.set_rng_state_all(checkpoint['torch_cuda_rng_state_all'])
        start_epoch = int(checkpoint['completed_epoch'])
        print(f'RESUME_CHECKPOINT={args.resume_checkpoint}')
        print(f'RESUME_FROM_COMPLETED_EPOCH={start_epoch}')
        print(f'NEXT_EPOCH={start_epoch + 1}')
        print(f'TARGET_EPOCH={args.epochs}')
        print(f'REMAINING_EPOCHS={args.epochs - start_epoch}')
"""
    train_text = train_text.replace('    best_test_metrics = model.init_metric_dict()\n    corpus.batch_size', '    best_test_metrics = model.init_metric_dict()\n' + resume_block + '    corpus.batch_size')
    train_text = train_text.replace('    for epoch in range(args.epochs):\n        model.train()', '    for epoch in range(start_epoch, args.epochs):\n        if epoch > start_epoch and epoch % args.checkpoint_every == 0:\n            _save_training_checkpoint(args, model, estimator, optimizer, optimizer_mi, lr_scheduler, best_test_metrics, corpus, epoch)\n        model.train()')
    train_text = train_text.replace("    print('Total time elapsed:", "    _save_training_checkpoint(args, model, estimator, optimizer, optimizer_mi, lr_scheduler, best_test_metrics, corpus, args.epochs)\n    print('Total time elapsed:")
    fixes.append({'file': 'train.py', 'before': 'single final model-only save', 'after': 'full-state atomic periodic checkpoints and exact resume support', 'reason': 'recover progress before Kaggle runtime termination', 'semantic_change': False})
    train_path.write_text(train_text)
patch = subprocess.run(["git", "-C", REPO_DIR, "diff"], text=True, stdout=subprocess.PIPE, check=True).stdout
Path(OUTPUT_DIR, "upstream_compatibility.patch").write_text(patch)
Path(OUTPUT_DIR, "source_fixes.json").write_text(json.dumps(fixes, indent=2))
Path(OUTPUT_DIR, "resume_audit.json").write_text(json.dumps({"failed_version": "previous-target-not-resumable", "last_completed_epoch": None, "outputs_downloaded": False, "checkpoint_found": False, "checkpoint_type": "NONE", "checkpoint_epoch": None, "exact_resume_possible": False, "reason": "No full-state checkpoint survived the failed Kaggle version; restart with checkpointing."}, indent=2))
print("SOURCE_FIXES=", len(fixes))
print("SOURCE_AUDIT=PASS")


# Harden the generated upstream patch. This cell runs inside Kaggle after the
# pinned source is checked out and before any training process starts.
import copy
import hashlib
import json
import os
import random

base_train = original_train
base_loader = original_loader
train_text = base_train
loader_text = base_loader
fixes = []

RESUME_CHECKPOINT_PATH = ""
resume_candidates = sorted(Path("/kaggle/input").glob("**/checkpoint_handoff.pt"))
if len(resume_candidates) > 1:
    raise RuntimeError("Ambiguous resume input: more than one checkpoint_handoff.pt was mounted")
if resume_candidates:
    RESUME_CHECKPOINT_PATH = str(resume_candidates[0])
    print("RESUME_INPUT_FOUND=" + Path(RESUME_CHECKPOINT_PATH).name)
elif RESUME_MODE == "FULL_STATE_RESUME":
    raise RuntimeError("FULL_STATE_RESUME requested but no checkpoint_handoff.pt was mounted")

if "return metrics, [att_s, att_i, att_t, att_mm]" in loader_text:
    if "y.byte()" in loader_text:
        loader_text = loader_text.replace("y.byte()", "y.bool()")
        fixes.append({"file": "utils/data_loader.py", "before": "y.byte()", "after": "y.bool()", "reason": "modern PyTorch boolean-mask compatibility", "semantic_change": False})

if "val_metrics = corpus.get_validation_pred(model, 'test')" in train_text:
    train_text = train_text.replace("val_metrics = corpus.get_validation_pred(model, 'test')", "val_metrics, _ = corpus.get_validation_pred(model, 'test')")
    fixes.append({"file": "train.py", "before": "val_metrics = corpus.get_validation_pred(model, 'test')", "after": "val_metrics, _ = corpus.get_validation_pred(model, 'test')", "reason": "upstream validation returns metrics and attention", "semantic_change": False})

train_text = train_text.replace("import argparse\n", "import argparse\nimport copy\nimport hashlib\nimport json\nimport os\nimport random\n", 1)
train_text = train_text.replace("        'save': 1,", "        'save': 1,\n        'resume_checkpoint': '',\n        'checkpoint_every': 25,\n        'checkpoint_dir': os.environ.get('MOMOK_CHECKPOINT_DIR', '/kaggle/working/repro_output/checkpoints'),\n        'session_soft_limit_seconds': 37800,")
train_text = train_text.replace("    if args.save:\n        torch.save(model.state_dict(), f'./checkpoint/{args.dataset}/{args.model}.pth')", "    if args.save:\n        os.makedirs(f'./checkpoint/{args.dataset}', exist_ok=True)\n        torch.save(model.state_dict(), f'./checkpoint/{args.dataset}/{args.model}.pth')")

helper = r"""

def _move_optimizer_state(optimizer, device):
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(device)


def _sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def _fsync_parent(path):
    try:
        fd = os.open(os.path.dirname(os.path.abspath(path)) or '.', os.O_RDONLY)
        try:
            os.fsync(fd)
        finally:
            os.close(fd)
    except OSError:
        pass


def _atomic_torch_save(payload, path):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    temporary = path + '.tmp'
    with open(temporary, 'wb') as handle:
        torch.save(payload, handle)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)
    _fsync_parent(path)
    digest = _sha256_file(path)
    with open(path + '.sha256', 'w') as handle:
        handle.write(digest + '  ' + os.path.basename(path) + '\n')
        handle.flush()
        os.fsync(handle.fileno())
    _fsync_parent(path + '.sha256')
    return digest


def _dataset_metadata():
    manifest_path = os.environ.get('MOMOK_DATASET_MANIFEST', '')
    if manifest_path and os.path.exists(manifest_path):
        with open(manifest_path) as handle:
            manifest = json.load(handle)
        canonical = json.dumps(manifest, sort_keys=True).encode()
        return {'manifest': manifest, 'fingerprint': hashlib.sha256(canonical).hexdigest()}
    return {'manifest': {}, 'fingerprint': ''}


def _checkpoint_payload(args, model, estimator, optimizer, optimizer_mi,
                       lr_scheduler, best_test_metrics, final_epoch_metrics,
                       corpus, completed_epoch):
    return {
        'format_version': 2,
        'completed_epoch': int(completed_epoch),
        'model_state_dict': model.state_dict(),
        'estimator_state_dict': estimator.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'optimizer_mi_state_dict': optimizer_mi.state_dict(),
        'lr_scheduler_state_dict': lr_scheduler.state_dict(),
        'best_test_metrics': copy.deepcopy(best_test_metrics),
        'final_epoch_metrics': copy.deepcopy(final_epoch_metrics),
        'python_random_state': random.getstate(),
        'numpy_rng_state': np.random.get_state(),
        'torch_rng_state': torch.get_rng_state(),
        'torch_cuda_rng_state_all': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
        'corpus_train_indices': copy.deepcopy(corpus.train_indices),
        'training_config': {
            'dataset': args.dataset, 'seed': args.seed, 'lr': args.lr,
            'mu': args.mu, 'dim': args.dim, 'r_dim': args.r_dim,
            'batch_size': args.batch_size, 'n_exp': args.n_exp,
            'epochs': args.epochs, 'eval_freq': args.eval_freq,
        },
        'source_commit': os.environ.get('MOMOK_SOURCE_COMMIT', ''),
        'dataset_metadata': _dataset_metadata(),
    }


def _save_training_checkpoint(args, model, estimator, optimizer, optimizer_mi,
                              lr_scheduler, best_test_metrics,
                              final_epoch_metrics, corpus, completed_epoch,
                              filename='checkpoint_latest.pt'):
    payload = _checkpoint_payload(args, model, estimator, optimizer, optimizer_mi,
                                  lr_scheduler, best_test_metrics,
                                  final_epoch_metrics, corpus, completed_epoch)
    path = os.path.join(args.checkpoint_dir, filename)
    digest = _atomic_torch_save(payload, path)
    if filename == 'checkpoint_latest.pt':
        print(f'CHECKPOINT_SAVED={filename} COMPLETED_EPOCH={completed_epoch} SHA256={digest}')
    return digest


def _load_full_checkpoint(path):
    # Explicit trusted full-state mode; arbitrary untrusted files are rejected.
    return torch.load(path, map_location='cpu', weights_only=False)
"""
train_text = train_text.replace('\n\ndef train_decoder(args):', helper + '\n\ndef train_decoder(args):', 1)

resume_block = r"""
    start_epoch = 0
    final_epoch_metrics = None
    if args.resume_checkpoint:
        checkpoint = _load_full_checkpoint(args.resume_checkpoint)
        required = [
            'format_version', 'completed_epoch', 'model_state_dict',
            'estimator_state_dict', 'optimizer_state_dict',
            'optimizer_mi_state_dict', 'lr_scheduler_state_dict',
            'best_test_metrics', 'final_epoch_metrics',
            'python_random_state', 'numpy_rng_state', 'torch_rng_state',
            'torch_cuda_rng_state_all', 'corpus_train_indices',
            'training_config', 'source_commit', 'dataset_metadata',
        ]
        missing = [key for key in required if key not in checkpoint]
        if missing:
            raise RuntimeError(f'Invalid full-state checkpoint; missing {missing}')
        expected = {
            'dataset': args.dataset, 'seed': args.seed, 'lr': args.lr,
            'mu': args.mu, 'dim': args.dim, 'r_dim': args.r_dim,
            'batch_size': args.batch_size, 'n_exp': args.n_exp,
            'epochs': args.epochs, 'eval_freq': args.eval_freq,
        }
        saved = checkpoint['training_config']
        for key, value in expected.items():
            if saved.get(key) != value:
                raise RuntimeError(f'Checkpoint config mismatch for {key}: {saved.get(key)!r} != {value!r}')
        if checkpoint['source_commit'] != os.environ.get('MOMOK_SOURCE_COMMIT', ''):
            raise RuntimeError('Checkpoint source commit mismatch')
        if checkpoint['dataset_metadata'].get('fingerprint') != _dataset_metadata().get('fingerprint'):
            raise RuntimeError('Checkpoint dataset inventory fingerprint mismatch')
        completed_epoch = int(checkpoint['completed_epoch'])
        if completed_epoch < 0 or completed_epoch >= args.epochs:
            raise RuntimeError(f'Invalid completed_epoch={completed_epoch}')
        model.load_state_dict(checkpoint['model_state_dict'])
        estimator.load_state_dict(checkpoint['estimator_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        optimizer_mi.load_state_dict(checkpoint['optimizer_mi_state_dict'])
        lr_scheduler.load_state_dict(checkpoint['lr_scheduler_state_dict'])
        _move_optimizer_state(optimizer, args.device)
        _move_optimizer_state(optimizer_mi, args.device)
        best_test_metrics = copy.deepcopy(checkpoint['best_test_metrics'])
        final_epoch_metrics = copy.deepcopy(checkpoint['final_epoch_metrics'])
        corpus.train_indices = copy.deepcopy(checkpoint['corpus_train_indices'])
        # Restore RNG only after all constructors and state loads.
        random.setstate(checkpoint['python_random_state'])
        np.random.set_state(checkpoint['numpy_rng_state'])
        torch.set_rng_state(checkpoint['torch_rng_state'])
        if torch.cuda.is_available():
            torch.cuda.set_rng_state_all(checkpoint['torch_cuda_rng_state_all'])
        start_epoch = completed_epoch
        print(f'RESUME_CHECKPOINT={args.resume_checkpoint}')
        print(f'RESUME_FROM_COMPLETED_EPOCH={completed_epoch}')
        print(f'NEXT_EPOCH={completed_epoch + 1}')
        print(f'TARGET_EPOCH={args.epochs}')
        print(f'REMAINING_EPOCHS={args.epochs - completed_epoch}')
"""
train_text = train_text.replace('    best_test_metrics = model.init_metric_dict()\n    corpus.batch_size', '    best_test_metrics = model.init_metric_dict()\n' + resume_block + '    corpus.batch_size', 1)

old_loop = """    training_range = tqdm(range(args.epochs))
    for epoch in training_range:
        model.train()
"""
new_loop = """    training_start = time.monotonic()
    training_range = tqdm(range(start_epoch, args.epochs))
    for epoch in training_range:
        model.train()
"""
if old_loop not in train_text:
    raise RuntimeError('Expected upstream training loop was not found')
train_text = train_text.replace(old_loop, new_loop, 1)

old_eval = """        if (epoch + 1) % args.eval_freq == 0:
            print("Epoch {:04d} , average loss {:.4f} , epoch_time {:.4f}\\n".format(
                epoch + 1, sum(epoch_loss) / len(epoch_loss), time.time() - t))
            model.eval()
            with torch.no_grad():
                val_metrics, _ = corpus.get_validation_pred(model, 'test')
            if val_metrics['MRR'] > best_test_metrics['MRR']:
                best_test_metrics['MRR'] = val_metrics['MRR']
            if val_metrics['MR'] < best_test_metrics['MR']:
                best_test_metrics['MR'] = val_metrics['MR']
            if val_metrics['Hits@1'] > best_test_metrics['Hits@1']:
                best_test_metrics['Hits@1'] = val_metrics['Hits@1']
            if val_metrics['Hits@3'] > best_test_metrics['Hits@3']:
                best_test_metrics['Hits@3'] = val_metrics['Hits@3']
            if val_metrics['Hits@10'] > best_test_metrics['Hits@10']:
                best_test_metrics['Hits@10'] = val_metrics['Hits@10']
            if val_metrics['Hits@100'] > best_test_metrics['Hits@100']:
                best_test_metrics['Hits@100'] = val_metrics['Hits@100']
            print('\\n'.join(['Epoch: {:04d}'.format(epoch + 1), model.format_metrics(val_metrics, 'test')]))
            print("\\n\\n")
"""
new_eval = """        completed_epoch = epoch + 1
        final_epoch_metrics = None
        if completed_epoch % args.eval_freq == 0:
            print("Epoch {:04d} , average loss {:.4f} , epoch_time {:.4f}\\n".format(
                completed_epoch, sum(epoch_loss) / len(epoch_loss), time.time() - t))
            model.eval()
            with torch.no_grad():
                val_metrics, _ = corpus.get_validation_pred(model, 'test')
            final_epoch_metrics = copy.deepcopy(val_metrics)
            for metric in ['MRR', 'Hits@1', 'Hits@3', 'Hits@10', 'Hits@100']:
                if val_metrics[metric] > best_test_metrics[metric]:
                    best_test_metrics[metric] = val_metrics[metric]
            if val_metrics['MR'] < best_test_metrics['MR']:
                best_test_metrics['MR'] = val_metrics['MR']
            print('\\n'.join(['Epoch: {:04d}'.format(completed_epoch), model.format_metrics(val_metrics, 'test')]))
            print("\\n\\n")
        if completed_epoch % args.checkpoint_every == 0 or final_epoch_metrics is not None or completed_epoch == args.epochs:
            _save_training_checkpoint(args, model, estimator, optimizer, optimizer_mi,
                                      lr_scheduler, best_test_metrics,
                                      final_epoch_metrics, corpus, completed_epoch)
        elapsed = time.monotonic() - training_start
        if elapsed >= args.session_soft_limit_seconds and completed_epoch < args.epochs:
            digest = _save_training_checkpoint(args, model, estimator, optimizer,
                                               optimizer_mi, lr_scheduler,
                                               best_test_metrics, final_epoch_metrics,
                                               corpus, completed_epoch,
                                               'checkpoint_handoff.pt')
            run_state = {
                'status': 'PARTIAL_CHECKPOINT_READY',
                'completed_epoch': completed_epoch,
                'target_epoch': args.epochs,
                'resume_required': True,
                'source_commit': os.environ.get('MOMOK_SOURCE_COMMIT', ''),
                'config': _checkpoint_payload(args, model, estimator, optimizer, optimizer_mi, lr_scheduler, best_test_metrics, final_epoch_metrics, corpus, completed_epoch)['training_config'],
                'checkpoint_filename': 'checkpoint_handoff.pt',
                'checkpoint_sha256': digest,
                'elapsed_seconds': elapsed,
                'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
            }
            with open(os.path.join(args.checkpoint_dir, 'run_state.json'), 'w') as handle:
                json.dump(run_state, handle, indent=2)
            print('RUN_STATE=PARTIAL_CHECKPOINT_READY')
            print(f'COMPLETED_EPOCH={completed_epoch}')
            return run_state
"""
if old_eval not in train_text:
    raise RuntimeError('Expected upstream evaluation block was not found')
train_text = train_text.replace(old_eval, new_eval, 1)

old_total = "    print('Total time elapsed: {:.4f}s'.format(time.time() - t_total))\n"
new_total = """    final_digest = _save_training_checkpoint(args, model, estimator, optimizer,
                                            optimizer_mi, lr_scheduler,
                                            best_test_metrics, final_epoch_metrics,
                                            corpus, args.epochs, 'checkpoint_final.pt')
    with open(os.path.join(args.checkpoint_dir, 'run_state.json'), 'w') as handle:
        json.dump({'status': 'FINAL_COMPLETE', 'completed_epoch': args.epochs,
                   'target_epoch': args.epochs, 'resume_required': False,
                   'source_commit': os.environ.get('MOMOK_SOURCE_COMMIT', ''),
                   'checkpoint_filename': 'checkpoint_final.pt',
                   'checkpoint_sha256': final_digest}, handle, indent=2)
    print('RUN_STATE=FINAL_COMPLETE')
    if final_epoch_metrics is not None:
        print('FINAL_EPOCH_METRICS_JSON=' + json.dumps(final_epoch_metrics, sort_keys=True))
    print('UPSTREAM_BEST_METRICS_JSON=' + json.dumps(best_test_metrics, sort_keys=True))
    print('Total time elapsed: {:.4f}s'.format(time.time() - t_total))
"""
if old_total not in train_text:
    raise RuntimeError('Expected upstream train tail was not found')
train_text = train_text.replace(old_total, new_total, 1)

train_path.write_text(train_text)
loader_path.write_text(loader_text)
patch = subprocess.run(['git', '-C', REPO_DIR, 'diff'], text=True, stdout=subprocess.PIPE, check=True).stdout
Path(OUTPUT_DIR, 'upstream_compatibility.patch').write_text(patch)
Path(OUTPUT_DIR, 'source_fixes.json').write_text(json.dumps(fixes + [{
    'file': 'train.py',
    'before': 'single model-only save and non-resumable loop',
    'after': 'full-state atomic checkpoints, graceful handoff, fail-closed resume',
    'reason': 'cross-session Kaggle durability',
    'semantic_change': False,
}], indent=2))
print('SOURCE_FIXES=', len(fixes) + 1)
print('CHECKPOINT_CONTRACT=PASS')
print('END_OF_EPOCH_ORDER=lr_scheduler.step() -> evaluation -> _save_training_checkpoint')
print('SOURCE_AUDIT=PASS')


In [ ]:
import glob, os, shutil, time, zipfile

import gdown
archive = Path(WORK_ROOT, "embeddings.zip")
last_error = None
for attempt in range(3):
    try:
        if archive.exists(): archive.unlink()
        gdown.download(id=GOOGLE_DRIVE_FILE_ID, output=str(archive), quiet=False)
        if archive.exists() and archive.stat().st_size > 1024:
            break
        raise RuntimeError("downloaded archive is missing or unexpectedly small")
    except Exception as exc:
        last_error = exc
        if attempt == 2: raise RuntimeError("official Google Drive embedding download failed") from exc
        time.sleep(2 ** attempt)

extract_dir = Path(WORK_ROOT, "embeddings_extracted")
if extract_dir.exists(): shutil.rmtree(extract_dir)
extract_dir.mkdir()
with zipfile.ZipFile(archive) as zf:
    zf.extractall(extract_dir)
    archive_names = zf.namelist()
files = [p for p in extract_dir.rglob("*") if p.is_file()]
print("embedding_archive_bytes=", archive.stat().st_size)
print("embedding_inventory=", [(str(p.relative_to(extract_dir)), p.stat().st_size) for p in files])
feature_members = FEATURE_ARCHIVE_MEMBERS
expected = {}
for name, member in feature_members.items():
    if archive_names.count(member) != 1:
        raise RuntimeError(f"Official archive must contain exact member once: {member}")
    expected[name] = extract_dir / Path(member)
    print(f"SELECTED_{name.replace('.pth', '').upper()}_MEMBER={member}")
dataset_dir = Path(REPO_DIR, "datasets", DATASET)
for name, source in expected.items():
    if not source.is_file(): raise RuntimeError(f"Extracted exact member is missing: {source}")
    shutil.copy2(source, dataset_dir / name)
print("MKG_W_EXACT_FEATURE_MEMBER_SELECTION=YES")
print("NO_FIRST_BASENAME_FEATURE_SELECTION=YES")
print("DATA_DOWNLOAD=PASS")

In [ ]:
import json, math, os

sys.path.insert(0, REPO_DIR)

dataset_dir = Path(REPO_DIR, "datasets", DATASET)
required = ["entity2id.txt", "relation2id.txt", "train.txt", "valid.txt", "test.txt", "img_features.pth", "text_features.pth"]
missing = [name for name in required if not (dataset_dir / name).exists()]
if missing: raise FileNotFoundError(missing)
entity_count = sum(1 for _ in open(dataset_dir / "entity2id.txt"))
relation_count = sum(1 for _ in open(dataset_dir / "relation2id.txt"))
triple_counts = {name: sum(1 for _ in open(dataset_dir / f"{name}.txt")) for name in ["train", "valid", "test"]}
img = torch.load(dataset_dir / "img_features.pth", map_location="cpu")
txt = torch.load(dataset_dir / "text_features.pth", map_location="cpu")
img_shape, txt_shape = list(img.shape), list(txt.shape)
assert img_shape[0] == entity_count and txt_shape[0] == entity_count
assert torch.isfinite(img).all() and torch.isfinite(txt).all()
manifest = {"dataset": DATASET, "entity_count": entity_count, "relation_count": relation_count, **{f"{k}_count": v for k, v in triple_counts.items()}, "image_shape": img_shape, "text_shape": txt_shape, "required_files": required}
Path(OUTPUT_DIR, "dataset_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
print("DATASET=PASS")

# MKG-W contract: raw availability differs from the entity-indexed tensors.
assert DATASET == "MKG-W"
assert entity_count == EXPECTED_DATASET["entities"]
assert relation_count == EXPECTED_DATASET["relations"]
assert triple_counts["train"] == EXPECTED_DATASET["train"]
assert triple_counts["valid"] == EXPECTED_DATASET["valid"]
assert triple_counts["test"] == EXPECTED_DATASET["test"]
assert img_shape == [EXPECTED_DATASET["entities"], EXPECTED_DATASET["image_dim"]]
assert txt_shape == [EXPECTED_DATASET["entities"], EXPECTED_DATASET["text_dim"]]
manifest.update({"raw_image_available": EXPECTED_DATASET["image_available"], "raw_text_available": EXPECTED_DATASET["text_available"], "official_entity_indexed_features": True})
Path(OUTPUT_DIR, "dataset_manifest.json").write_text(json.dumps(manifest, indent=2))
print("ACTIVE_DATASET_MKG_W=PASS")
print("FEATURE_CONTRACT=PASS")
DATASET_MANIFEST_PATH = str(Path(OUTPUT_DIR, "dataset_manifest.json"))
os.environ["MOMOK_DATASET_MANIFEST"] = DATASET_MANIFEST_PATH


In [ ]:
import os

WANDB_ENABLED = False
WANDB_STATUS = "NOT_CONFIGURED"
WANDB_URL = None
try:
    from kaggle_secrets import UserSecretsClient
    secret = UserSecretsClient().get_secret("WANDB_API_KEY")
    if secret:
        os.environ["WANDB_API_KEY"] = secret
        WANDB_ENABLED = True
        WANDB_STATUS = "CONFIGURED"
except Exception:
    pass
Path(OUTPUT_DIR, "wandb_status.json").write_text(json.dumps({"enabled": WANDB_ENABLED, "status": WANDB_STATUS, "url": WANDB_URL}, indent=2))
print("WANDB_STATUS=" + WANDB_STATUS)

In [ ]:
import os, re

WANDB_RUN = None
if os.environ.get("WANDB_API_KEY"):
    import wandb
    WANDB_RUN = wandb.init(
        project=os.environ.get("WANDB_PROJECT", "graphml-momok-reproduction"),
        entity=os.environ.get("WANDB_ENTITY") or None,
        group="main",
        job_type="reproduction",
        config={"dataset": DATASET, "seed": SEED, "epochs": EPOCHS, "tracking_mode": "live_kaggle"},
        reinit="finish_previous",
    )
    WANDB_RUN.define_metric("epoch")
    WANDB_RUN.define_metric("train/*", step_metric="epoch")
    WANDB_RUN.define_metric("eval/*", step_metric="epoch")
    WANDB_RUN.define_metric("runtime/*", step_metric="epoch")

_WANDB_EPOCH = None
def wandb_log_training_line(line):
    global _WANDB_EPOCH
    if WANDB_RUN is None:
        return
    epoch_match = re.search(r"Epoch\s+(\d+)\s*,\s*average loss\s+([-+]?\d+(?:\.\d+)?)\s*,\s*epoch_time\s+([-+]?\d+(?:\.\d+)?)", line)
    if epoch_match:
        _WANDB_EPOCH = int(epoch_match.group(1))
        main_loss = re.search(r"loss=main:\s*([-+]?\d+(?:\.\d+)?)", line)
        mi_loss = re.search(r"mi:\s*([-+]?\d+(?:\.\d+)?)", line)
        record = {"epoch": _WANDB_EPOCH, "train/loss": float(epoch_match.group(2)), "runtime/epoch_time_sec": float(epoch_match.group(3))}
        if mi_loss:
            record["train/mi_estimator_loss"] = float(mi_loss.group(1))
        WANDB_RUN.log(record)
    eval_match = re.search(r"test_Hits@10:\s*([-+]?\d+(?:\.\d+)?)\s+test_Hits@3:\s*([-+]?\d+(?:\.\d+)?)\s+test_Hits@1:\s*([-+]?\d+(?:\.\d+)?)\s+test_MR:\s*([-+]?\d+(?:\.\d+)?)\s+test_MRR:\s*([-+]?\d+(?:\.\d+)?)", line)
    if eval_match and _WANDB_EPOCH is not None:
        WANDB_RUN.log({"epoch": _WANDB_EPOCH, "eval/MRR": float(eval_match.group(5)), "eval/Hits@1": float(eval_match.group(3)), "eval/Hits@3": float(eval_match.group(2)), "eval/Hits@10": float(eval_match.group(1)), "eval/MR": float(eval_match.group(4))})

print("WANDB_LIVE_SUPPORT=READY")

In [ ]:
import re, subprocess, time

smoke_log = Path(OUTPUT_DIR, "smoke.log")
smoke_cmd = [sys.executable, "-u", "train.py", "--cuda", "0", "--dataset", DATASET, "--lr", str(LR), "--mu", str(MU), "--dim", str(DIM), "--batch_size", str(BATCH_SIZE), "--n_exp", str(N_EXPERTS), "--seed", str(SEED), "--epochs", "2", "--eval_freq", "1", "--save", "0", "--checkpoint_every", "25", "--checkpoint_dir", str(Path(OUTPUT_DIR, "checkpoints"))]
run_env = os.environ.copy()
run_env["MOMOK_SOURCE_COMMIT"] = REPO_COMMIT
run_env["MOMOK_CHECKPOINT_DIR"] = str(Path(OUTPUT_DIR, "checkpoints"))
start = time.time()
result = subprocess.run(smoke_cmd, cwd=REPO_DIR, env=run_env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
smoke_log.write_text(result.stdout)
print(result.stdout)
checks = {
    "ENVIRONMENT": result.returncode == 0 and torch.cuda.is_available(),
    "GPU": torch.cuda.is_available(),
    "DATASET": "Training data" in result.stdout,
    "MODEL_INIT": "Total number of parameters" in result.stdout,
    "FORWARD": "average loss" in result.stdout,
    "BACKWARD": "average loss" in result.stdout,
    "MI_ESTIMATOR_STEP": "mi:" in result.stdout,
    "EVALUATION": "test_MRR" in result.stdout,
    "MRR_PRESENT": "test_MRR" in result.stdout,
    "HITS_PRESENT": all(k in result.stdout for k in ["test_Hits@1", "test_Hits@3", "test_Hits@10"]),
    "NAN_INF": not bool(re.search(r"(?i)\b(?:nan|inf)\b", result.stdout)),
}
summary = {"status": "PASS" if all(checks.values()) else "FAIL", "checks": checks, "command": smoke_cmd, "wall_time_sec": time.time() - start}
Path(OUTPUT_DIR, "smoke_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
if not all(checks.values()): raise RuntimeError("Smoke test failed; full training was not launched")

checkpoint_test = Path(OUTPUT_DIR, "checkpoints", "checkpoint_final.pt")
if checkpoint_test.exists():
    payload = torch.load(checkpoint_test, map_location="cpu", weights_only=False)
    required_checkpoint_keys = {
        "format_version", "completed_epoch", "model_state_dict", "estimator_state_dict",
        "optimizer_state_dict", "optimizer_mi_state_dict", "lr_scheduler_state_dict",
        "best_test_metrics", "final_epoch_metrics", "python_random_state", "numpy_rng_state",
        "torch_rng_state", "torch_cuda_rng_state_all", "corpus_train_indices",
        "training_config", "source_commit", "dataset_metadata",
    }
    assert required_checkpoint_keys.issubset(payload), sorted(required_checkpoint_keys - set(payload))
    assert Path(str(checkpoint_test) + ".sha256").exists()
    print("CHECKPOINT_ROUNDTRIP=PASS")
else:
    raise RuntimeError("Smoke test did not produce checkpoint_final.pt")

# Independent runtime round-trip using the actual Kaggle PyTorch runtime.
import copy
import hashlib
import random
import types

runtime_dir = Path(OUTPUT_DIR, "checkpoint_runtime_test")
runtime_dir.mkdir(parents=True, exist_ok=True)
class _RuntimeNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = torch.nn.Linear(4, 3)

    def forward(self, x):
        return self.layer(x)

runtime_model = _RuntimeNet().to("cuda")
runtime_estimator = _RuntimeNet().to("cuda")
runtime_optimizer = torch.optim.Adam(runtime_model.parameters(), lr=0.001)
runtime_optimizer_mi = torch.optim.Adam(runtime_estimator.parameters(), lr=0.001)
runtime_scheduler = torch.optim.lr_scheduler.ExponentialLR(runtime_optimizer, 1.0)
runtime_input = torch.randn(2, 4, device="cuda")
runtime_loss = runtime_model(runtime_input).sum() + runtime_estimator(runtime_input).sum()
runtime_loss.backward()
runtime_optimizer.step()
runtime_optimizer_mi.step()
runtime_scheduler.step()
runtime_corpus_order = [{"triple": (1, 2, 3), "label": [3]}]
random.seed(1234)
numpy.random.seed(1234)
torch.manual_seed(1234)
torch.cuda.manual_seed_all(1234)
runtime_payload = {
    "format_version": 2,
    "completed_epoch": 2,
    "model_state_dict": copy.deepcopy(runtime_model.state_dict()),
    "estimator_state_dict": copy.deepcopy(runtime_estimator.state_dict()),
    "optimizer_state_dict": copy.deepcopy(runtime_optimizer.state_dict()),
    "optimizer_mi_state_dict": copy.deepcopy(runtime_optimizer_mi.state_dict()),
    "lr_scheduler_state_dict": copy.deepcopy(runtime_scheduler.state_dict()),
    "best_test_metrics": {"MRR": 0.1},
    "final_epoch_metrics": {"MRR": 0.1},
    "python_random_state": random.getstate(),
    "numpy_rng_state": numpy.random.get_state(),
    "torch_rng_state": torch.get_rng_state(),
    "torch_cuda_rng_state_all": torch.cuda.get_rng_state_all(),
    "corpus_train_indices": copy.deepcopy(runtime_corpus_order),
    "training_config": {"dataset": "MKG-W", "seed": 10010, "lr": 0.001, "mu": 0.0001, "dim": 200, "r_dim": 256, "batch_size": 1024, "n_exp": 3, "epochs": 2000, "eval_freq": 100},
    "source_commit": REPO_COMMIT,
    "dataset_metadata": {"fingerprint": "runtime-test"},
}
runtime_path = runtime_dir / "runtime_roundtrip.pt"
with runtime_path.open("wb") as handle:
    torch.save(runtime_payload, handle)
    handle.flush()
    os.fsync(handle.fileno())
runtime_digest = hashlib.sha256(runtime_path.read_bytes()).hexdigest()
(runtime_path.with_suffix(runtime_path.suffix + ".sha256")).write_text(runtime_digest + "  " + runtime_path.name + "\n")
assert hashlib.sha256(runtime_path.read_bytes()).hexdigest() == runtime_digest
print("CHECKPOINT_SHA256_VERIFY=PASS")
loaded_runtime = torch.load(runtime_path, map_location="cpu", weights_only=False)
required_runtime_keys = {"model_state_dict", "estimator_state_dict", "optimizer_state_dict", "optimizer_mi_state_dict", "lr_scheduler_state_dict", "python_random_state", "numpy_rng_state", "torch_rng_state", "torch_cuda_rng_state_all", "corpus_train_indices"}
assert required_runtime_keys.issubset(loaded_runtime)
print("CHECKPOINT_FULL_STATE_LOAD=PASS")
runtime_model.load_state_dict(loaded_runtime["model_state_dict"])
runtime_estimator.load_state_dict(loaded_runtime["estimator_state_dict"])
runtime_optimizer.load_state_dict(loaded_runtime["optimizer_state_dict"])
runtime_optimizer_mi.load_state_dict(loaded_runtime["optimizer_mi_state_dict"])
runtime_scheduler.load_state_dict(loaded_runtime["lr_scheduler_state_dict"])
runtime_corpus_order = copy.deepcopy(loaded_runtime["corpus_train_indices"])
random.setstate(loaded_runtime["python_random_state"])
numpy.random.set_state(loaded_runtime["numpy_rng_state"])
torch.set_rng_state(loaded_runtime["torch_rng_state"])
torch.cuda.set_rng_state_all(loaded_runtime["torch_cuda_rng_state_all"])
assert runtime_corpus_order == [{"triple": (1, 2, 3), "label": [3]}]
print("CHECKPOINT_RUNTIME_ROUNDTRIP=PASS")

# Bounded control-flow proof for completed_epoch=N -> next human epoch=N+1.
control_completed_epoch = 2
control_target_epoch = 5
control_resumed_zero_based = list(range(control_completed_epoch, control_target_epoch))
control_resumed_human = [index + 1 for index in control_resumed_zero_based]
assert control_resumed_human == [3, 4, 5]
assert control_resumed_human[0] != control_completed_epoch
assert control_resumed_human[-1] == control_target_epoch
print("RESUME_NEXT_EPOCH_TEST=PASS")
print("RESUME_NO_REPLAY=PASS")
print("RESUME_NO_SKIP=PASS")
print("CONTROL_FLOW_EQUIVALENCE=PASS")
print("BITWISE_TRAJECTORY_EQUIVALENCE=NOT_GUARANTEED")
print("V3_RUNTIMENET_FORWARD_REGRESSION=PASS")

print("MKG_W_RUNTIME_SMOKE=PASS")

In [ ]:
gpu_before = run_capture(["bash", "-lc", "nvidia-smi --query-gpu=name,driver_version,memory.total,memory.used,utilization.gpu --format=csv,noheader"])
print(gpu_before.stdout)
Path(OUTPUT_DIR, "gpu_before.json").write_text(json.dumps({"nvidia_smi": gpu_before.stdout}, indent=2))

In [ ]:
import re, time

full_cmd = [sys.executable, "-u", "train.py", "--cuda", "0", "--dataset", DATASET, "--lr", str(LR), "--mu", str(MU), "--dim", str(DIM), "--batch_size", str(BATCH_SIZE), "--n_exp", str(N_EXPERTS), "--seed", str(SEED), "--epochs", str(EPOCHS), "--eval_freq", str(EVAL_FREQ), "--save", "1", "--checkpoint_every", "25", "--checkpoint_dir", str(Path(OUTPUT_DIR, "checkpoints")), "--session_soft_limit_seconds", str(SESSION_SOFT_LIMIT_SECONDS)]
if globals().get("RESUME_CHECKPOINT_PATH"):
    full_cmd += ["--resume_checkpoint", RESUME_CHECKPOINT_PATH]
print("FULL_COMMAND=", " ".join(full_cmd))
start_time = time.time()
log_path = Path(OUTPUT_DIR, "mkgw_full.log")
with log_path.open("w") as log_file:
    process = subprocess.Popen(full_cmd, cwd=REPO_DIR, env=run_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
        log_file.write(line)
        wandb_log_training_line(line)
    return_code = process.wait()
wall_time_sec = time.time() - start_time
Path(OUTPUT_DIR, "full_run_meta.json").write_text(json.dumps({"command": full_cmd, "return_code": return_code, "wall_time_sec": wall_time_sec}, indent=2))
run_state_path = Path(OUTPUT_DIR, "checkpoints", "run_state.json")
run_state = json.loads(run_state_path.read_text()) if run_state_path.exists() else {}
PARTIAL_RUN = run_state.get("status") == "PARTIAL_CHECKPOINT_READY"
if PARTIAL_RUN:
    Path(OUTPUT_DIR, "PARTIAL_HANDOFF.md").write_text(
        "# Partial MoMoK MKG-W Handoff\n\n"
        f"Training stopped cleanly at completed epoch {run_state['completed_epoch']}. "
        "Resume is required from checkpoint_handoff.pt. No final verdict was generated.\n"
    )
elif return_code != 0:
    raise RuntimeError(f"Full MKG-W run failed with exit code {return_code}")

In [ ]:
import json, math, re

if globals().get("PARTIAL_RUN", False):
    print("PARTIAL_HANDOFF=PASS")
    print("No final metrics or reproduction verdict is generated for a partial segment.")
else:
    log_text = Path(OUTPUT_DIR, "mkgw_full.log").read_text()
    final_match = re.findall(r"FINAL_EPOCH_METRICS_JSON=(\{.*?\})", log_text)
    best_match = re.findall(r"UPSTREAM_BEST_METRICS_JSON=(\{.*?\})", log_text)
    if not final_match or not best_match:
        raise RuntimeError("Missing explicit final metric markers")
    final_epoch_metrics = json.loads(final_match[-1])
    upstream_best_metrics = json.loads(best_match[-1])
    reproduced = {name: float(final_epoch_metrics[name]) for name in PAPER_METRICS}
    best = {name: float(upstream_best_metrics[name]) for name in PAPER_METRICS}
    if not all(math.isfinite(v) for v in reproduced.values()):
        raise RuntimeError("Final metrics contain NaN/Inf")
    absolute_delta = {k: reproduced[k] - PAPER_METRICS[k] for k in PAPER_METRICS}
    relative_delta = {k: absolute_delta[k] / PAPER_METRICS[k] for k in PAPER_METRICS}
    max_abs = max(abs(v) for v in absolute_delta.values())
    verdict = "STRONG_MATCH" if max_abs <= 0.01 else "REPRO_MATCH" if max_abs <= 0.02 else "OUTSIDE_EXPECTED_RANGE"
    run_state = json.loads(Path(OUTPUT_DIR, "checkpoints", "run_state.json").read_text())
    if run_state.get("status") != "FINAL_COMPLETE" or run_state.get("completed_epoch") != 2000:
        raise RuntimeError("Final completion state is not exactly epoch 2000")
    full_meta = json.loads(Path(OUTPUT_DIR, "full_run_meta.json").read_text())
    summary = {
        "experiment": {"research_question": "RQ1", "paper_section": "5.3", "paper_table": "Table 1", "dataset": DATASET, "model": "MoMoK Full Model", "ablation": False},
        "final_epoch": 2000,
        "final_epoch_metrics": reproduced,
        "upstream_best_metrics": best,
        "paper_targets": PAPER_METRICS,
        "paper": PAPER_METRICS,
        "reproduced": reproduced,
        "absolute_delta": absolute_delta,
        "relative_delta": relative_delta,
        "source_commit": REPO_COMMIT,
        "hyperparameter_source": CONFIG_SOURCE,
        "config": {"dataset": DATASET, "seed": SEED, "lr": LR, "mu": MU, "dim": DIM, "batch_size": BATCH_SIZE, "n_experts": N_EXPERTS, "epochs": EPOCHS, "eval_freq": EVAL_FREQ, "command": full_meta["command"]},
        "environment": environment,
        "wall_time_sec": full_meta["wall_time_sec"],
        "wandb": json.loads(Path(OUTPUT_DIR, "wandb_status.json").read_text()),
        "verdict": verdict,
    }
    Path(OUTPUT_DIR, "repro_summary.json").write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2))


## Final human-readable report

The next cell writes `repro_output/REPRODUCTION_REPORT.md`, including the source pin, environment, dataset manifest, exact command, paper comparison, verdict, optional W&B state, compatibility fixes, and known limitations.

In [ ]:
if globals().get("PARTIAL_RUN", False):
    print("PARTIAL_REPORT=SKIPPED_FINAL_VERDICT_NOT_ALLOWED")
else:
    import json

    manifest = json.loads(Path(OUTPUT_DIR, "dataset_manifest.json").read_text())
    summary = json.loads(Path(OUTPUT_DIR, "repro_summary.json").read_text())
    fixes = json.loads(Path(OUTPUT_DIR, "source_fixes.json").read_text())
    report = f"""# MoMoK MKG-W Reproduction

    ## SOURCE
    - Repository: {{REPO_URL}}
    - Commit: `{{REPO_COMMIT}}`

    ## ENVIRONMENT
    - Python: {{environment['effective_python']}}
    - PyTorch: {{environment['effective_torch']}}
    - CUDA: {{environment['effective_cuda']}}
    - GPU: {{environment['gpu']}}
    - Runtime deviation: {{environment['runtime_deviation']}}

    ## DATASET
    - Entities: {{manifest['entity_count']}}
    - Relations: {{manifest['relation_count']}}
    - Train / valid / test: {{manifest['train_count']}} / {{manifest['valid_count']}} / {{manifest['test_count']}}
    - Image shape: {{manifest['image_shape']}}
    - Text shape: {{manifest['text_shape']}}

    ## CONFIG
    - Exact command: `{{' '.join(summary['config']['command'])}}`
    - Seed: {{SEED}}
    - lr / mu / dim: {{LR}} / {{MU}} / {{DIM}}
    - Batch size / epochs: {{BATCH_SIZE}} / {{EPOCHS}}
    - Hyperparameter source: {{CONFIG_SOURCE}} (the official MoMoK README command)

    ## PAPER
    - MRR: 0.3589
    - H@1: 0.3038
    - H@3: 0.3754
    - H@10: 0.4613

    ## REPRODUCED
    - MRR: {{summary['reproduced']['MRR']:.4f}}
    - H@1: {{summary['reproduced']['Hits@1']:.4f}}
    - H@3: {{summary['reproduced']['Hits@3']:.4f}}
    - H@10: {{summary['reproduced']['Hits@10']:.4f}}

    ## DELTA
    - MRR: {{summary['absolute_delta']['MRR']:.6f}}
    - H@1: {{summary['absolute_delta']['Hits@1']:.6f}}
    - H@3: {{summary['absolute_delta']['Hits@3']:.6f}}
    - H@10: {{summary['absolute_delta']['Hits@10']:.6f}}

    ## VERDICT
    `{{summary['verdict']}}` using the reproduction acceptance bands defined by this notebook.

    ## WANDB
    - Status: {{summary['wandb']['status']}}
    - Run URL: {{summary['wandb'].get('url')}}

    ## SOURCE COMPATIBILITY FIXES
    {{json.dumps(fixes, indent=2)}}

    ## KNOWN LIMITATIONS
    - The official repository history does not contain a verified MKG-W launch script; the candidate configuration is explicitly inferred from the supplied reproduction contract.
    - Kaggle runtime versions may differ from the paper's Python 3.9 / PyTorch 2.0.0 environment.
    - This notebook runs the main full model only; it does not run ablations or PMF.
    """
    Path(OUTPUT_DIR, "REPRODUCTION_REPORT.md").write_text(report)
    print(report)

In [ ]:
for path in sorted(Path(OUTPUT_DIR).rglob("*")):
    if path.is_file(): print(f"{path.relative_to(OUTPUT_DIR)}\t{path.stat().st_size} bytes")
print("OUTPUT_INVENTORY=PASS")